# Notebook 7: Information Extraction - Unstructured Text → Knowledge Graph

The final notebook demonstrates the reverse direction: using an LLM to extract structured RDF triples from unstructured policy documents, guided by the ontology.


---
### Setup


In [ ]:
%graph_notebook_config --store-to NOTEBOOK_CONFIG --silent

In [ ]:
from workshop import *
import json
configure(neptune_config=json.loads(NOTEBOOK_CONFIG))

# Load ontology text for extraction prompts
ontology_ttl = (NOTEBOOK_DIR / "ontology.ttl").read_text()
import boto3
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
ont = load_ontology()

---
## Module 10: Information Extraction - Unstructured Text → Knowledge Graph

So far, all our data has been **structured** - hand-authored Turtle files with precise triples.
In practice, much of the information that feeds a knowledge graph arrives as **unstructured text**:
policy documents, agent notes, intake forms, correspondence.

This module demonstrates **ontology-guided information extraction**:
1. Start with a natural-language policy document
2. Provide the ontology as a schema constraint to the LLM
3. Have the LLM extract entities, relationships, and attributes as RDF triples (Turtle)
4. Validate and load the extracted triples into Neptune

The ontology acts as a **semantic contract** - the LLM can only produce triples that conform
to the classes and properties we've defined. This prevents hallucinated predicates and ensures
the extracted graph is structurally compatible with existing data.

In [ ]:
# ── 10.1: Sample Policy Document ─────────────────────────────────────
# This simulates an unstructured document that an intake system might receive.
# It contains entities (Customer, Policy, Coverage, Claims, Events) embedded
# in natural language — exactly the kind of text we want to extract from.

POLICY_DOCUMENT = """# Policy Summary — P004 (HSA)

**Policyholder:** Max Premium (Customer ID: CUST-004)

Greenfield Benefits Group issued Health Savings Account policy P004 to Max Premium,
effective January 15, 2025. The policy is classified as an HSA plan.

## Coverage History

The original coverage terms provided a coverage limit of USD 120,000 with a
USD 1,500 deductible and a monthly premium of USD 575. This coverage was effective
from January 15, 2025 through December 31, 2025.

Following the annual review, updated coverage took effect on January 1, 2026.
The revised terms increased the coverage limit to USD 140,000, maintained the
USD 1,500 deductible, and adjusted the monthly premium to USD 610. This coverage
is currently active with no scheduled end date.

## Claim Activity

On November 3, 2025, Max Premium filed claim C006 for USD 8,500 related to an
emergency room visit on October 28, 2025. The claim was submitted on November 3, 2025
with the full amount of USD 8,500 requested. Following review, the claim was approved
on November 10, 2025 and payment of USD 8,500 was issued on November 18, 2025.

A second claim, C007, was submitted on March 15, 2026 for USD 2,200 stemming from
a specialist consultation on March 10, 2026. This claim is currently pending review.
"""

display(Markdown(POLICY_DOCUMENT))

### Step 2: Ontology-Guided Extraction

We send the unstructured document to Bedrock along with the ontology as context. The LLM extracts structured RDF triples in Turtle format, guided by the ontology's classes and properties. The ontology acts as an extraction schema - the LLM knows exactly what entities and relationships to look for:


In [ ]:
# ── 10.2: Ontology-Guided Extraction via Bedrock ────────────────────
# We send the ontology + document to the most capable Bedrock model and
# ask it to extract structured RDF triples in Turtle format.

EXTRACTION_PROMPT = """You are an RDF knowledge graph extraction engine. Given an ontology and an unstructured document, extract all entities, relationships, and attributes as RDF triples in Turtle format.

ONTOLOGY (defines the ONLY classes and properties you may use):

```turtle
{ontology}
```

RULES:
- Use ONLY classes and properties defined in the ontology above
- Use the namespace prefix : for <http://example.org/insurance#>
- Every entity MUST have an rdf:type statement
- Respect domain/range constraints from the ontology
- Use proper XSD datatypes: xsd:string, xsd:decimal, xsd:dateTime, xsd:boolean
- Format dateTime values as ISO 8601: "YYYY-MM-DDThh:mm:ss"^^xsd:dateTime
- Use URI-safe identifiers for instances (e.g., :policyP004, :claimC006, :customerCUST004)
- Connect entities via object properties (:forPolicy, :hasEvent, :hasCoverage, :heldBy)
- For PolicyCoverage, set :isCurrent true for active coverage, false for expired
- Return ONLY valid Turtle — no explanation, no markdown fences, no comments

DOCUMENT:
{document}

TURTLE:"""


def extract_triples(document: str, ontology: str) -> str:
    """Extract RDF triples from unstructured text using Bedrock LLM."""
    prompt = EXTRACTION_PROMPT.format(ontology=ontology, document=document)

    response = bedrock.invoke_model(
        modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 4096,
            "temperature": 0,
            "messages": [{"role": "user", "content": prompt}],
        }),
    )

    result = json.loads(response["body"].read())
    turtle_output = result["content"][0]["text"].strip()

    # Strip markdown fences if the model wraps them
    if turtle_output.startswith("```"):
        turtle_output = turtle_output.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    return turtle_output


# ── Fallback Turtle (used when Bedrock is unavailable) ──────────────
FALLBACK_EXTRACTED_TTL = """@prefix : <http://example.org/insurance#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:customerCUST004 a :Customer ;
    :customerId "CUST-004" ;
    :customerName "Max Premium" .

:policyP004 a :Policy ;
    :policyId "P004" ;
    :policyType "HSA" ;
    :heldBy :customerCUST004 ;
    :hasCoverage :covP004v1, :covP004v2 .

:covP004v1 a :PolicyCoverage ;
    :coverageAmount 120000.00 ;
    :deductible 1500.00 ;
    :premium 575.00 ;
    :validFrom "2025-01-15T00:00:00"^^xsd:dateTime ;
    :validTo "2025-12-31T23:59:59"^^xsd:dateTime ;
    :isCurrent false .

:covP004v2 a :PolicyCoverage ;
    :coverageAmount 140000.00 ;
    :deductible 1500.00 ;
    :premium 610.00 ;
    :validFrom "2026-01-01T00:00:00"^^xsd:dateTime ;
    :isCurrent true .

:claimC006 a :Claim ;
    :claimId "C006" ;
    :claimAmount 8500.00 ;
    :incidentDate "2025-10-28T00:00:00"^^xsd:dateTime ;
    :forPolicy :policyP004 ;
    :hasEvent :evtC006sub, :evtC006app, :evtC006paid .

:evtC006sub a :ClaimEvent ;
    :eventType "submitted" ;
    :eventTimestamp "2025-11-03T00:00:00"^^xsd:dateTime ;
    :eventAmount 8500.00 .

:evtC006app a :ClaimEvent ;
    :eventType "approved" ;
    :eventTimestamp "2025-11-10T00:00:00"^^xsd:dateTime .

:evtC006paid a :ClaimEvent ;
    :eventType "paid" ;
    :eventTimestamp "2025-11-18T00:00:00"^^xsd:dateTime ;
    :eventAmount 8500.00 .

:claimC007 a :Claim ;
    :claimId "C007" ;
    :claimAmount 2200.00 ;
    :incidentDate "2026-03-10T00:00:00"^^xsd:dateTime ;
    :forPolicy :policyP004 ;
    :hasEvent :evtC007sub .

:evtC007sub a :ClaimEvent ;
    :eventType "submitted" ;
    :eventTimestamp "2026-03-15T00:00:00"^^xsd:dateTime ;
    :eventAmount 2200.00 .
"""

# ── Run extraction ──────────────────────────────────────────────────
print("Extracting triples from policy document...\n")

extracted_ttl = extract_triples(POLICY_DOCUMENT, ontology_ttl)
if extracted_ttl:
    source_label = "Bedrock (Claude Sonnet 4.5)"
else:
    extracted_ttl = FALLBACK_EXTRACTED_TTL
    source_label = "Fallback (pre-built)"

print(f"Source: {source_label}\n")
display(Markdown(f"```turtle\n{extracted_ttl}\n```"))

### Step 3: Validate Extracted Triples

The LLM-extracted triples are parsed and validated against the ontology. This catches extraction errors - wrong property names, invalid types, or relationships that don't conform to the domain model. The ontology serves as a quality gate for LLM output:


In [ ]:
# ── 10.3: Validate Extracted Triples Against Ontology ────────────────
# Parse the extracted Turtle and verify every class and property exists
# in the ontology. This catches LLM hallucinations (invented predicates).

from rdflib import Graph as RdfGraph, Namespace, RDF, OWL

extracted_graph = RdfGraph()
try:
    extracted_graph.parse(data=extracted_ttl, format="turtle")
    print(f"Parsed {len(extracted_graph)} triples from extracted Turtle\n")
except Exception as e:
    print(f"Parse error: {e}")
    print("Falling back to pre-built extraction...")
    extracted_graph = RdfGraph()
    extracted_graph.parse(data=FALLBACK_EXTRACTED_TTL, format="turtle")
    print(f"Parsed {len(extracted_graph)} triples from fallback\n")

# Validate: every rdf:type must reference an ontology class
# and every predicate must be defined in the ontology
INS_NS = Namespace("http://example.org/insurance#")
ont_classes = set(ont.subjects(RDF.type, OWL.Class))
ont_data_props = set(ont.subjects(RDF.type, OWL.DatatypeProperty))
ont_obj_props = set(ont.subjects(RDF.type, OWL.ObjectProperty))
ont_all_props = ont_data_props | ont_obj_props

errors = []
for s, p, o in extracted_graph:
    # Check rdf:type targets
    if p == RDF.type and o != OWL.Ontology:
        if o not in ont_classes:
            errors.append(f"Unknown class: {o}")
    # Check predicates in our namespace
    elif str(p).startswith(str(INS_NS)):
        if p not in ont_all_props:
            errors.append(f"Unknown property: {p}")

if errors:
    print(f"⚠️  {len(errors)} validation error(s):")
    for e in errors:
        print(f"  → {e}")
else:
    print("✅ All classes and properties conform to the ontology")

# Summary statistics
entities = set()
for s in extracted_graph.subjects(RDF.type, None):
    for t in extracted_graph.objects(s, RDF.type):
        entities.add((s, t))

print(f"\nExtracted entities:")
for entity, rdf_type in sorted(entities, key=lambda x: str(x[1])):
    label = str(entity).replace(str(INS_NS), ":")
    cls = str(rdf_type).replace(str(INS_NS), ":")
    print(f"  {label:<24} a {cls}")

# Count relationships (object properties)
rel_count = sum(1 for _, p, o in extracted_graph
                if p in ont_obj_props)
attr_count = sum(1 for _, p, o in extracted_graph
                 if p in ont_data_props)
print(f"\nRelationships: {rel_count}  |  Attributes: {attr_count}  |  Total triples: {len(extracted_graph)}")

### Step 4: Persist to Neptune

Validated triples are loaded into Neptune, extending the knowledge graph with information extracted from unstructured text. This closes the loop: unstructured documents → LLM extraction → ontology validation → persistent knowledge graph:


In [ ]:
# ── 10.4: Persist Extracted Triples to Neptune ──────────────────────
# Load the extracted graph into Neptune so it's queryable alongside
# the existing claims data. Uses the same SigV4 auth as Module 5.

# Serialize to N-Triples for Neptune INSERT DATA
ntriples = extracted_graph.serialize(format="nt")
# Wrap in SPARQL INSERT DATA
insert_query = f"INSERT DATA {{\n{ntriples}\n}}"

print("Loading extracted triples into Neptune...")
try:
    status = neptune_update(insert_query)
    print(f"  HTTP {status} — {len(extracted_graph)} triples loaded\n")

    # Verify: query back the new policy
    rows = sparql_query("""
        PREFIX : <http://example.org/insurance#>
        SELECT ?policyId ?policyType ?customerName ?coverageAmount ?isCurrent
        WHERE {
            ?policy a :Policy ; :policyId ?policyId ; :policyType ?policyType ;
                    :heldBy ?customer ; :hasCoverage ?cov .
            ?customer :customerName ?customerName .
            ?cov :coverageAmount ?coverageAmount ; :isCurrent ?isCurrent .
            FILTER (?policyId = "P004")
        }
        ORDER BY ?isCurrent
    """)

    if rows:
        print("Verification — Policy P004 in Neptune:\n")
        print(f"  {'Policy':<10} {'Type':<6} {'Customer':<20} {'Coverage':>12} {'Current'}")
        print(f"  {'─' * 62}")
        for r in rows:
            current = r['isCurrent']
            print(f"  {r['policyId']:<10} {r['policyType']:<6} {r['customerName']:<20} ${float(r['coverageAmount']):>11,.2f} {current}")
    else:
        print("  (No results — policy may need a moment to index)")
except Exception as e:
    print(f"  Neptune load failed: {e}")
    print("  Showing N-Triples that would be loaded:\n")
    print(ntriples[:1000])


---
## Summary

### What You Built

**Pipeline 1 - Extract (Module 10):** Unstructured text → structured RDF via ontology-guided LLM

```
  Unstructured Document          Ontology (OWL)
         │                            │
         ▼                            ▼
       ┌────────────────────────────────┐
       │          Bedrock LLM           │
       │       (text → RDF extract)     │
       └──────────────┬─────────────────┘
                      │
               Extracted Triples
                      │
                      ▼
       ┌────────────────────────────────┐
       │         Graph Backend          │
       │  Neptune / Ontop VKG / rdflib  │
       └────────────────────────────────┘
```

**Pipeline 2 - Query & Explain (Modules 7–9):** Natural language → SPARQL → SHACL → denial explanation

```
  Natural Language Question
         │
         ▼
  ┌──────────────┐      ┌───────────────┐      ┌─────────────────────────────┐
  │  Bedrock LLM │─────▶│ SPARQL Query  │─────▶│        Graph Backend        │
  │ (text→SPARQL)│      └───────────────┘      │ Neptune / Ontop VKG / rdflib│
  └──────────────┘                             └──────────────┬──────────────┘
                                                              │
                                                         Claim Data
                                                              │
                                                              ▼
                                                      ┌────────────────┐
                                                      │  SHACL Shapes  │
                                                      │   (pyshacl)    │
                                                      └───────┬────────┘
                                                              │
                                                          Violations
                                                              │
                                                              ▼
                                                      ┌────────────────┐
                                                      │  Bedrock LLM   │
                                                      │(explain denial)│
                                                      └───────┬────────┘
                                                              │
                                                              ▼
                                                       Customer-Facing
                                                      Denial Explanation
```

### Key Takeaways

1. **Ontology as schema** - OWL defines the vocabulary; SHACL enforces constraints
2. **SCD2 in RDF** - Temporal coverage history enables gap detection without SQL
3. **Business rules as shapes** - Eligibility logic is declarative, testable, and auditable
4. **No hardcoded denial codes** - SHACL violations ARE the denial reasons
5. **LLM as translator** - Bedrock converts machine-readable violations into human language
6. **Triple backend** - Same SPARQL queries work against Neptune, Ontop VKG, or rdflib
7. **Virtual Knowledge Graph** - Ontop exposes relational data as RDF with zero ETL
8. **Ontology-guided extraction** - LLM extracts structured RDF from unstructured text, constrained by the ontology to prevent hallucinated predicates